In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
import pickle
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, accuracy_score
from torch.cuda.amp import autocast, GradScaler
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
DATA_DIR = "../data"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"A usar dispositivo: {device}")

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, "dataset_treino.csv"))
df_val = pd.read_csv(os.path.join(DATA_DIR, "dataset_validacao.csv"))

# Remover qualquer linha que tenha texto ou label vazios antes de fazer mais nada
df_train = df_train.dropna(subset=['Text', 'Label'])
df_val = df_val.dropna(subset=['Text', 'Label'])

# Garantir que é tudo texto
df_train['Text'] = df_train['Text'].astype(str)
df_val['Text'] = df_val['Text'].astype(str)


# Modelo 1 - Classificação Binária

Human vs AI
  - Só classifica se é humano ou AI

# 1. Mapeamento Binário

In [ ]:
# Se for 'Human', fica 'Human'. Se for qualquer outra coisa, vira 'AI'.
df_train['Label_Bin'] = np.where(df_train['Label'] == 'Human', 'Human', 'AI')
df_val['Label_Bin'] = np.where(df_val['Label'] == 'Human', 'Human', 'AI')

# O nosso novo mapeamento super simples
label_mapping_binario = {'AI': 0, 'Human': 1}
print("Mapeamento Binário:", label_mapping_binario)

df_train['Label_Bin'] = np.where(df_train['Label'].str.strip() == 'Human', 'Human', 'AI')
df_val['Label_Bin'] = np.where(df_val['Label'].str.strip() == 'Human', 'Human', 'AI')

df_train['label'] = df_train['Label_Bin'].map(label_mapping_binario).fillna(0).astype(int)
df_val['label'] = df_val['Label_Bin'].map(label_mapping_binario).fillna(0).astype(int)

df_train['label'] = df_train['Label_Bin'].map(label_mapping_binario)
df_val['label'] = df_val['Label_Bin'].map(label_mapping_binario)

# 2. Tokenizador e Dataset 

In [ ]:

from transformers import AutoModelForSequenceClassification
MODEL_NAME = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=200): 
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_length)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # AQUI: A correção do torch.long para a Loss não explodir
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

BATCH_SIZE = 32
train_loader = DataLoader(TextDataset(df_train['Text'], df_train['label'], tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TextDataset(df_val['Text'], df_val['label'], tokenizer), batch_size=BATCH_SIZE*2)

# 3. Arquitetura do Modelo Binario

In [ ]:
print("A carregar modelo nativo de classificação...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
).to(device)

# 4. Configurar Treino

In [ ]:

EPOCHS = 3 
LEARNING_RATE = 5e-5


optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=0.01,
    eps=1e-6
)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=50,  
    num_training_steps=total_steps
)

# 5. Loop de Treino e Validação (Modelo 1)

In [ ]:
print("A iniciar o treino do Modelo 1 (HuggingFace Native)...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device) 

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss
        logits = outputs.logits
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()

        if (i+1) % 100 == 0:
            print(f"Época {epoch+1} | Batch {i+1}/{len(train_loader)} | Loss Média: {total_loss/(i+1):.4f}")

    # --- Validação ---
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"\n✅ Época {epoch+1} Concluída - Média de Loss: {total_loss/len(train_loader):.4f}")
    print(f"🎯 Validação (Humano vs IA): Acurácia = {val_acc:.4f}\n")
    print(classification_report(val_labels, val_preds, target_names=['AI (0)', 'Human (1)']))

# 8. Guardar o Modelo Binario

In [ ]:
torch.save(model.state_dict(), 'modelo_binario_human_vs_ai.pt')
with open('label_mapping_binario.pkl', 'wb') as f:
    pickle.dump(label_mapping_binario, f)
    
print("Treino Binário concluído e modelo guardado com sucesso!")

## Modelo 2 - Classificação Multiclasse

# 1. Preparar Dados EXCLUSIVOS DE IA

In [ ]:
print("A filtrar apenas os textos gerados por Inteligência Artificial...")

# Remover os Humanos do treino e da validação
df_train_ai = df_train[df_train['Label'].str.strip() != 'Human'].copy()
df_val_ai = df_val[df_val['Label'].str.strip() != 'Human'].copy()

# Criar o novo mapeamento (Só 4 classes agora)
unique_ai_labels = sorted(df_train_ai['Label'].dropna().unique().tolist())
label_mapping_ai = {k: v for v, k in enumerate(unique_ai_labels)}
reverse_mapping_ai = {v: k for k, v in label_mapping_ai.items()}
print("\nNovo Mapeamento do Detetive (Multiclasse):")
print(label_mapping_ai)

# Aplicar o mapeamento
df_train_ai['label'] = df_train_ai['Label'].map(label_mapping_ai).astype(int)
df_val_ai['label'] = df_val_ai['Label'].map(label_mapping_ai).astype(int)

# 2. Tokenizador e Novo Dataset

In [ ]:
MODEL_NAME = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True) 

class AIDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=200): 
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_length)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

BATCH_SIZE = 32
train_loader_ai = DataLoader(AIDataset(df_train_ai['Text'], df_train_ai['label'], tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader_ai = DataLoader(AIDataset(df_val_ai['Text'], df_val_ai['label'], tokenizer), batch_size=BATCH_SIZE*2)

print(f"\nDados prontos! O novo treino tem {len(train_loader_ai)} batches por época.")

# 3. Arquitetura do Modelo MultiClasse

In [ ]:
print("A carregar modelo nativo para Multiclasse (4 IAs)...")
model_ai = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=4  
).to(device)

# 4. Configurar Treino

In [ ]:
EPOCHS_AI = 4 
LEARNING_RATE_AI = 5e-5

optimizer_ai = torch.optim.AdamW(
    model_ai.parameters(), 
    lr=LEARNING_RATE_AI, 
    weight_decay=0.01,
    eps=1e-6
)

total_steps_ai = len(train_loader_ai) * EPOCHS_AI
scheduler_ai = get_linear_schedule_with_warmup(
    optimizer_ai, 
    num_warmup_steps=50, 
    num_training_steps=total_steps_ai
)

# 5. Loop de Treino e Validação (Modelo 2)

In [ ]:
print("A iniciar o treino do Modelo 2 (Detetive das IAs)...")
for epoch in range(EPOCHS_AI):
    model_ai.train()
    total_loss = 0

    for i, batch in enumerate(train_loader_ai):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device) 

        optimizer_ai.zero_grad()

        outputs = model_ai(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss
        logits = outputs.logits
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ai.parameters(), max_norm=1.0)
        
        optimizer_ai.step()
        scheduler_ai.step()
        
        total_loss += loss.item()

        if (i+1) % 50 == 0: # Imprime a cada 50 batches porque este dataset é mais pequeno
            print(f"Época {epoch+1} | Batch {i+1}/{len(train_loader_ai)} | Loss Média: {total_loss/(i+1):.4f}")

    # --- Validação do Modelo 2 ---
    model_ai.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader_ai:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model_ai(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"\n✅ Época {epoch+1} Concluída - Média de Loss: {total_loss/len(train_loader_ai):.4f}")
    print(f"🎯 Validação (Só IA): Acurácia = {val_acc:.4f}\n")
    
    # Mostrar o relatório com os nomes corretos das IAs
    target_names = [reverse_mapping_ai[i] for i in range(len(label_mapping_ai))] if 'reverse_mapping_ai' in locals() else list(label_mapping_ai.keys())
    print(classification_report(val_labels, val_preds, target_names=target_names))

# 6. Guardar o Modelo 2

In [ ]:
torch.save(model_ai.state_dict(), 'modelo_multiclasse_so_ai.pt')

with open('label_mapping_ai.pkl', 'wb') as f:
    pickle.dump(label_mapping_ai, f)
    
print("Treino Multiclasse concluído e Modelo 2 guardado com sucesso!")